## GENeSYS-MOD RES Tool - GERMANY / country-method verification

Country-selection workflow (regions=['DE']) with CORINE land cover, to verify the refactored functions still work with the original method. Requires `geodata/corine.tif`.

## General requirements & preparation

This timeseries script uses functions from atlite to generate weather timeseries for GENeSYS-MOD. It uses the ERA5 weather dataset, which is downloaded and transformed locally.
Please be aware that for the entire European dataset, a large amount of system memory is required (at least 24GB of RAM).


To use this script, you need to have the following packages installed:
- numpy
- matplotlib
- seaborn
- pandas / geopandas
- scikit-learn
- cartopy
- xarray
- atlite

In addition, in order to be able to load the ERA5 cutouts, you need an API key from https://cds.climate.copernicus.eu/user/register?destination=%2F%23!%2Fhome

To activate the API key, follow the instructions at https://cds.climate.copernicus.eu/api-how-to

For more information on how to run atlite, please refer to https://atlite.readthedocs.io/en/latest/

In [ ]:
from functions import *

In [ ]:
onshore_turbine = 'Vestas_V112_3MW'
offshore_turbine = 'Vestas_V164_7MW_offshore'
solar_panel = 'CSi'

# COUNTRY method (the original workflow): regions are ISO-A2 country codes and
# geo_file is a natural-earth admin-1 file carrying iso_a2 / iso_3166_2 columns.
# This verifies the refactored functions still work with the old country path.
geo_file = "geodata/natural_earth_world_admin1.geojson"
offshore_file = None      # country method: no custom EEZ here
bathymetry_file = None

timeframe = "2018"
filename = "germany"
output_dir = create_output_folder(timeframe)

admin = 0                 # 0 = country level (DE); 1 = admin-1 / states
regions = ["DE"]          # ISO-A2 codes

cutout_north_west = []
cutout_south_east = []

# --- Run switch (speed) -------------------------------------------------
# Skip the full-area capacity-factor timeseries block (PV/wind/heat) while
# iterating on the GIS potentials; set True for a full timeseries export.
generate_full_area_timeseries = False


In [ ]:
# PV slope/azimuth for utility-scale (ground-mounted) PV.
pv_slope = 36.7
pv_azimuth = 180

# Optimal tilt: if True, utility-scale PV uses a per-coordinate optimal fixed
# tilt (function of latitude) and equator-facing azimuth instead of the single
# pv_slope/pv_azimuth above. Cheap closed-form rule (no per-angle yield search).
pv_optimal_tilt = True

# Rooftop PV is installed differently (roof pitch / mixed orientations), so it
# keeps its own fixed slope/azimuth for the separate pv_rooftop timeseries.
rooftop_pv_slope = 25
rooftop_pv_azimuth = 180

# Usable-site thresholds: minimum developable share of a cutout cell for it
# to count as a usable location for the capacity-factor timeseries. Land is
# lenient (1%); rooftop is the built-up fraction (small everywhere), so it
# needs a higher bar to drop near-empty cells - raise if still too many.
usable_threshold = 0.01
usable_threshold_rooftop = 0.05

In [ ]:
# Here, you need to define the horizontal and vertical distances (in degrees) between your coordinates
dx_step = 0.275
dy_step = 0.275

## Region Defitinition & Cutout preparation

In this block, you need to define your coordinates, either by using your own fixed coordinates (by defining custom bounds), or using the automatic .geojson mapping for the country of your choice.

Be aware that the first preparation of the cutout will take significant time!

In [ ]:
# Pre-check cutout size before downloading. Country method derives bounds from
# the ISO codes in 'regions' via natural earth.
estimate_cutout_size(timeframe, regions=regions, dx=dx_step, dy=dy_step)

In [ ]:
cutout = get_cutout(filename,
                    timeframe,
                    regions=regions,
                    geo_file=geo_file,
                    dx=dx_step,
                    dy=dy_step)

Plotting your cutout to ensure that everything is correct!

You can add zoom=True/False to zoom into your selection and use size to set the size of the graph if needed.

In [ ]:
plot_country_map(cutout) # plot_country_map(cutout,zoom=False,size=8)

## Functions for timeseries generation

In [ ]:
coords_onshore, coords_offshore = get_coords(cutout, 
                                             regions, 
                                             geo_file, 
                                             admin=admin,
                                             offshore_file=offshore_file,
                                             bathymetry_file=bathymetry_file)

In [ ]:
# Controlled by 'generate_full_area_timeseries' switch (set in the config cell).
if generate_full_area_timeseries:
    pv_inf, pv_avg, pv_opt = pv_capacity_factors(cutout, 
                                                 coords_onshore, 
                                                 solar_panel,
                                                 pv_slope=pv_slope,
                                                 pv_azimuth=pv_azimuth, 
                                                 optimal_tilt=pv_optimal_tilt,
                                                 timeframe=timeframe, 
                                                 filename=filename,
                                                 write_raw_data=False,
                                                 output_dir=output_dir)

In [ ]:
# Controlled by 'generate_full_area_timeseries' switch (set in the config cell).
if generate_full_area_timeseries:
    pv_tra = pv_capacity_factors(cutout, 
                                 coords_onshore, 
                                 solar_panel,
                                 tracking= "horizontal",
                                 pv_azimuth=0, #refers to oriantation of the axis in this case
                                 timeframe=timeframe, 
                                 filename=filename,
                                 write_raw_data=False,
                                 output_dir=output_dir)

In [ ]:
# Controlled by 'generate_full_area_timeseries' switch (set in the config cell).
if generate_full_area_timeseries:
    display(pv_inf.mean())
    display(pv_avg.mean())
    display(pv_opt.mean())

In [ ]:
# Controlled by 'generate_full_area_timeseries' switch (set in the config cell).
if generate_full_area_timeseries:
    display(pv_tra.mean())

In [ ]:
# Controlled by 'generate_full_area_timeseries' switch (set in the config cell).
if generate_full_area_timeseries:
    wind_onshore_inf, wind_onshore_avg, wind_onshore_opt = wind_onshore_capacity_factors(cutout, 
                                                                                         coords_onshore, 
                                                                                         onshore_turbine,  
                                                                                         timeframe=timeframe, 
                                                                                         filename=filename,
                                                                                         write_raw_data=False, # determines if you want to have the aggregated outputs per administrative region or the full dataset of each individual coordinate (default: False)
                                                                                         output_dir=output_dir)

In [ ]:
# Controlled by 'generate_full_area_timeseries' switch (set in the config cell).
if generate_full_area_timeseries:
    display(wind_onshore_inf.mean()) 
    display(wind_onshore_avg.mean()) 
    display(wind_onshore_opt.mean())

In [ ]:
# Controlled by 'generate_full_area_timeseries' switch (set in the config cell).
if generate_full_area_timeseries:
    wind_offshore_shallow, wind_offshore_transitional, wind_offshore_deep = wind_offshore_capacity_factors(cutout, 
                                                                                                           coords_offshore, 
                                                                                                           offshore_turbine, 
                                                                                                           timeframe=timeframe,
                                                                                                           filename=filename,
                                                                                                           write_raw_data=False, # determines if you want to have the aggregated outputs per administrative region or the full dataset of each individual coordinate (default: False)
                                                                                                           output_dir=output_dir)

In [ ]:
# Controlled by 'generate_full_area_timeseries' switch (set in the config cell).
if generate_full_area_timeseries:
    display(wind_offshore_shallow.mean())
    display(wind_offshore_transitional.mean())
    display(wind_offshore_deep.mean())

In [ ]:
# Controlled by 'generate_full_area_timeseries' switch (set in the config cell).
if generate_full_area_timeseries:
    df_heatpump_ground_cop, df_cooling, df_heating, df_heatpump_cop = temperature_timeseries(cutout,
                                                                                            coords_onshore,
                                                                                            timeframe=timeframe,
                                                                                            filename=filename,
                                                                                            write_raw_data=False, # determines if you want to have the aggregated outputs per administrative region or the full dataset of each individual coordinate (default: False)
                                                                                            output_dir=output_dir)

In [ ]:
# Controlled by 'generate_full_area_timeseries' switch (set in the config cell).
if generate_full_area_timeseries:
    display(df_heatpump_ground_cop.mean())
    display(df_heatpump_cop.mean())
    display(df_cooling.mean())
    display(df_heating.mean())

### GIS-based renewable energy potentials

##### This part is still somewhat experimental. It uses functionality from the Atlite package to calculate available capcities for utility-scale PV, onshore wind, and rooftop PV installations.
##### It uses the information from the timeseries part to compute the shapefiles, the only thing that needs to be provided is the excluder zones (e.g. WDPA for protected areas). CORINE is provided for land cover in Europe, anything else would need to be separately downloaded.

In [ ]:
# Country method: build shapes from ISO codes via natural earth.
shapes, regions_name_en = gis_get_country_geometry(regions, admin, cutout)

In [ ]:
CORINE = "geodata/corine.tif"
WDPA = "geodata/WDPA_Oct2024_Public_shp-polygons.shp"

# corine.tif is int8 (nodata -128). atlite add_raster defaults nodata=255 which
# overflows int8 -> "Cannot convert fill_value 255 to dtype int8". Pass nodata=-128.
excluder = ExclusionContainer()
excluder.add_raster(CORINE, codes=range(20), nodata=-128)
#excluder.add_geometry(WDPA)

cities = ExclusionContainer()
cities.add_raster(CORINE, codes=range(1,6), invert=True, nodata=-128)

In [ ]:
pv_cap_per_sqkm = 100                   # MW
pv_percent_land_available = 0.03        # % of suitable area
wind_cap_per_sqkm = 27                  # MW
wind_percent_land_available = 0.03      # % of suitable area
rooftop_cap_per_sqkm = 100              # MW
rooftop_percent_area_available = 0.2    # % of all rooftops

In [ ]:
AvailabilityMatrix = calculate_and_plot_available_area(admin, cutout, shapes, regions_name_en, excluder)

In [ ]:
AvailabilityMatrix_Rooftop = calculate_and_plot_available_rooftops(admin, cutout, shapes, regions_name_en, cities)

In [ ]:
output_df = calculate_capacity_potentials(cutout, coords_onshore, AvailabilityMatrix, AvailabilityMatrix_Rooftop,
                                          pv_cap_per_sqkm, pv_percent_land_available,
                                          wind_cap_per_sqkm, wind_percent_land_available,
                                          rooftop_cap_per_sqkm, rooftop_percent_area_available)

In [ ]:
# here is the resulting DataFrame containing all the potentials and areas that have been calculated
output_df